# BaanDee Living: Messy Company to Working MVP

Instructor-led walkthrough using synthetic data. Follow the numbered cells with the Class 1 slides. The application is read-only.

In [ ]:
# Tool boundary: Python loads the project; helper code remains in reusable modules.
from pathlib import Path
import json, sys, subprocess
import pandas as pd
import requests

ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
RAW = ROOT / "data" / "raw"
CP = ROOT / "data" / "checkpoints"
print("Project root:", ROOT)
print("Synthetic fixture mode is the default; no secret is displayed.")

## Inspect before cleaning
Which defects can change the customer response?

In [ ]:
files = ["customers.csv", "orders.csv", "inventory_snapshot.csv", "support_tickets.csv"]
profile = []
for name in files:
    df = pd.read_csv(RAW / name, dtype=str)
    profile.append({"file": name, "rows": len(df), "columns": len(df.columns), "missing_cells": int(df.isna().sum().sum())})
display(pd.DataFrame(profile))
tickets_raw = pd.read_csv(RAW / "support_tickets.csv", dtype=str)
display(tickets_raw.loc[tickets_raw.ticket_id.isin(["TKT-260731-042", "TKT-260731-099", "TKT-260731-117"])])

## A clean-looking join can still fail
The ticket uses an alias from another system.

In [ ]:
orders_raw = pd.read_csv(RAW / "orders.csv", dtype=str)
naive = tickets_raw.merge(orders_raw, left_on="order_reference", right_on="order_id", how="left")
display(naive.loc[naive.ticket_id == "TKT-260731-042", ["ticket_id", "order_reference", "order_id", "export_status"]])

In [ ]:
from src.data_prep import unsafe_digits_only_match
matches = unsafe_digits_only_match("OD1842", orders_raw)
print("Digit-only/suffix matches:", matches)
print("Conclusion: digits alone cannot prove which order belongs to the ticket.")

## Conservative correction
Use known formats and an approved source crosswalk; retain ambiguity.

In [ ]:
from src.data_prep import normalize_customer_id, normalize_order_reference, build_checkpoints, GROUND_TRUTH
crosswalk = pd.read_csv(GROUND_TRUTH / "order_reference_crosswalk.csv", dtype=str)
print('customer:', normalize_customer_id(' cus00842 '))
print('order:', normalize_order_reference('OD1842', crosswalk))
report = build_checkpoints()
print('join quality:', report)

In [ ]:
corrected = pd.read_csv(CP / "02_corrected_join.csv", dtype=str)
display(corrected.loc[corrected.ticket_id == "TKT-260731-042", ["ticket_id", "order_reference", "order_id_clean", "order_id", "export_status"]])
print("Quarantined date records:")
display(pd.read_csv(CP / "quarantine_dates.csv", dtype=str)[["source", "order_date", "created_at"]].fillna(""))
print("Ambiguous customer candidates:")
display(pd.read_csv(CP / "quarantine_customer_candidates.csv", dtype=str)[["customer_id", "customer_name", "email", "phone"]])

## A real HTTP boundary
FastAPI owns the current operational fixture. `requests` crosses the boundary and returns HTTP status plus JSON.

In [ ]:
# Important client lines; the API must be running in another terminal.
url = "http://127.0.0.1:8000/orders/ORD-260728-1842"
response = requests.get(url, timeout=2)
print("URL:", url)
print("HTTP status:", response.status_code)
display(response.json())

In [ ]:
from src.api_client import fetch_order
valid = fetch_order("ORD-260728-1842")
unknown = fetch_order("ORD-UNKNOWN-77")
unavailable = fetch_order("ORD-260728-1842", base_url="http://127.0.0.1:8999", timeout=0.2)
display(pd.DataFrame([
    {"case":"valid", "status":valid.status_code, "ok":valid.ok, "error":valid.error},
    {"case":"unknown", "status":unknown.status_code, "ok":unknown.ok, "error":unknown.error},
    {"case":"unavailable", "status":unavailable.status_code, "ok":unavailable.ok, "error":unavailable.error},
]))

## Evidence before language
Current API status wins over an older export, but both remain visible.

In [ ]:
from src.evidence import build_case_evidence
evidence = build_case_evidence("TKT-260731-042")
display({
    "ticket_id": evidence["ticket_id"],
    "order_id": evidence["order_id"],
    "export_status": evidence["order_export"]["export_status"],
    "exported_at": evidence["order_export"]["exported_at"],
    "current_status": evidence["current_order"]["status"],
    "current_as_of": evidence["current_order"]["as_of"],
    "estimated_delivery": evidence["estimated_delivery"],
    "source_decision": evidence["source_decision"],
})
print("Deterministic baseline:\n", evidence["rule_response"])

## What does AI add?
The model receives a bounded evidence object. Fixture mode is synthetic and does not prove live model quality.

In [ ]:
from src.ai_adapter import build_prompt, draft_ai_response
from src.evidence import check_grounding
print(build_prompt(evidence))
ai_text, source = draft_ai_response(evidence, mode="fixture")
print("\nSource:", source)
print(ai_text)
print("Grounding check:", check_grounding(ai_text, evidence))

## From button to business logic
Gradio supplies the interface; the event calls the same functions used above.

In [ ]:
from src.app import load_case_for_ui
summary, policy, rule, ai, evaluation = load_case_for_ui("TKT-260731-099", "fixture")
display(summary)
print(rule)
display(evaluation)
print("UI binding in src/app.py: button.click(fn=load_case_for_ui, inputs=[ticket, mode], outputs=[...])")

## A failure becomes a safeguard
The contained demo fails; the normal suite protects the corrected behavior.

In [ ]:
demo = subprocess.run([sys.executable, str(ROOT / "tests" / "demo_failure.py")], cwd=ROOT, capture_output=True, text=True)
print(demo.stdout)
print(demo.stderr.splitlines()[-1])
assert demo.returncode != 0, "The contained unsafe-match demonstration should fail"
suite = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=ROOT, capture_output=True, text=True)
print(suite.stdout)
assert suite.returncode == 0, suite.stderr

## Working MVP, known limits

The application can assemble evidence and prepare a grounded draft. It still lacks real authentication, write-action approval, systematic AI evaluation, production monitoring, and operational validation. Those questions begin Class 2.